In [ ]:
# ============================================================================
# ALL TEXTURES COMBINED - Generate all textures in one cell
# ============================================================================
import numpy as np

# Define common dimensions
num_channels = 4
value_range = (0, 10)
z_range = (0, 48)
y_range = (0, 343)
x_range = (0, 680)

num_values = value_range[1] - value_range[0] + 1
z_dim = z_range[1] - z_range[0] + 1
y_dim = y_range[1] - y_range[0] + 1
x_dim = x_range[1] - x_range[0] + 1

# Calculate scaling factors
x_scale = x_dim / 172
y_scale = y_dim / 87

# ============================================================================
# Helper Functions
# ============================================================================

def create_sine_strip(data, channel, y_center, strip_size, amplitude, period, z_dim, y_dim, x_dim):
    """Create a sine wave strip pattern in x direction."""
    strip_half = strip_size // 2
    for z in range(z_dim):
        for x in range(x_dim):
            y_sine = y_center + amplitude * np.sin(2 * np.pi * x / period)
            y_sine_int = int(round(y_sine))
            y_start = max(0, y_sine_int - strip_half)
            y_end = min(y_dim, y_sine_int + strip_half + 1)
            for y in range(y_start, y_end):
                random_value = np.random.randint(0, 11)
                data[channel, random_value, z, y, x] = 1

def create_line_strip(data, channel, y_center, strip_size, z_dim, y_dim, x_dim):
    """Create a straight line strip in x direction at y = y_center."""
    strip_half = strip_size // 2
    y_start = max(0, y_center - strip_half)
    y_end = min(y_dim, y_center + strip_half + 1)
    for z in range(z_dim):
        for x in range(x_dim):
            for y in range(y_start, y_end):
                random_value = np.random.randint(0, 11)
                data[channel, random_value, z, y, x] = 1

def create_random_cloud(center_x, center_y, center_z, radius, z_dim, y_dim, x_dim):
    """Create a random cloud of points around a center with random shape"""
    num_points = np.random.randint(20, 101)
    shape_type = np.random.choice(['sphere', 'ellipsoid', 'elongated'])
    points = []
    for _ in range(num_points):
        if shape_type == 'sphere':
            theta = np.random.uniform(0, 2 * np.pi)
            phi = np.random.uniform(0, np.pi)
            r = np.random.uniform(0, radius) ** (1/3)
            dx = r * np.sin(phi) * np.cos(theta)
            dy = r * np.sin(phi) * np.sin(theta)
            dz = r * np.cos(phi)
        elif shape_type == 'ellipsoid':
            axes = np.random.uniform(0.3, 1.0, 3) * radius
            theta = np.random.uniform(0, 2 * np.pi)
            phi = np.random.uniform(0, np.pi)
            r = np.random.uniform(0, 1) ** (1/3)
            dx = r * axes[0] * np.sin(phi) * np.cos(theta)
            dy = r * axes[1] * np.sin(phi) * np.sin(theta)
            dz = r * axes[2] * np.cos(phi)
        else:  # elongated
            direction = np.random.uniform(-1, 1, 3)
            direction = direction / np.linalg.norm(direction)
            r_parallel = np.random.uniform(0, radius)
            r_perp = np.random.uniform(0, radius * 0.4)
            perp1 = np.cross(direction, [1, 0, 0])
            if np.linalg.norm(perp1) < 0.1:
                perp1 = np.cross(direction, [0, 1, 0])
            perp1 = perp1 / np.linalg.norm(perp1)
            perp2 = np.cross(direction, perp1)
            perp2 = perp2 / np.linalg.norm(perp2)
            angle = np.random.uniform(0, 2 * np.pi)
            offset = r_parallel * direction + r_perp * (np.cos(angle) * perp1 + np.sin(angle) * perp2)
            dx, dy, dz = offset[0], offset[1], offset[2]
        
        x = int(center_x + dx)
        y = int(center_y + dy)
        z = int(center_z + dz)
        if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z < z_dim:
            points.append((x, y, z))
    return points

def create_striped_circle(data, channel, center_x, center_y, max_radius, stripe_width, z_dim, y_dim, x_dim):
    """Create a circle with a single stripe/ring."""
    inner_radius = max_radius - stripe_width
    outer_radius = max_radius
    for z in range(z_dim):
        for y in range(y_dim):
            for x in range(x_dim):
                dx = x - center_x
                dy = y - center_y
                distance = np.sqrt(dx**2 + dy**2)
                if inner_radius <= distance <= outer_radius:
                    random_value = np.random.randint(0, 11)
                    data[channel, random_value, z, y, x] = 1

def create_concentrated_3d_cloud_point(center_x, center_y, center_z, radius_xy, radius_z, num_points, z_dim, x_dim, y_dim):
    """Create a concentrated 3D cloud of points around a center"""
    cloud_points = []
    for _ in range(num_points):
        angle = np.random.uniform(0, 2 * np.pi)
        r_xy = radius_xy * np.sqrt(np.random.uniform(0, 1))
        x = int(center_x + r_xy * np.cos(angle))
        y = int(center_y + r_xy * np.sin(angle))
        dz = int(np.random.uniform(-radius_z, radius_z))
        z = center_z + dz
        z = int(np.clip(z, 0, z_dim - 1))
        if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z < z_dim:
            cloud_points.append((x, y, z))
    return cloud_points

# ============================================================================
# TEXTURE 1: Sinusoidal Pattern
# ============================================================================
def generate_sinusoid_texture():
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    y_center = 100
    strip_size = 20
    sine_params = [(0, 100, 400), (1, 50, 200), (2, 150, 300)]
    for channel, amplitude, period in sine_params:
        create_sine_strip(data, channel, y_center, strip_size, amplitude, period, z_dim, y_dim, x_dim)
    create_line_strip(data, 3, y_center, strip_size, z_dim, y_dim, x_dim)
    return data

# ============================================================================
# TEXTURE 2: Colonies Pattern
# ============================================================================
def generate_colonies_texture():
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    max_radius = 50
    num_colonies_per_line = 8
    
    # Channel 0: 3D Arc
    num_arc_points = num_colonies_per_line * 3
    t_values = np.linspace(0, 1, num_arc_points)
    arc_amplitude = 0.3
    
    for t in t_values:
        x_arc_base = t * (x_dim - 1)
        y_arc_base = t * (y_dim - 1)
        z_arc_base = t * (z_dim - 1)
        angle = t * np.pi
        arc_elevation_y = arc_amplitude * (y_dim - 1) * np.sin(angle)
        arc_elevation_z = arc_amplitude * (z_dim - 1) * np.sin(angle)
        x_arc = int(np.clip(x_arc_base, 0, x_dim - 1))
        y_arc = int(np.clip(y_arc_base + arc_elevation_y, 0, y_dim - 1))
        z_arc = int(np.clip(z_arc_base + arc_elevation_z, 0, z_dim - 1))
        radius = np.random.uniform(10, max_radius)
        cloud_points = create_random_cloud(x_arc, y_arc, z_arc, radius, z_dim, y_dim, x_dim)
        for x, y, z_val in cloud_points:
            if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z_val < z_dim:
                random_value = np.random.randint(6, 11)
                data[0, random_value, z_val, y, x] = 1
    
    # Generate remaining channels
    for i in range(0, 3):
        for z in range(z_dim):
            # Channel 0: Concave curve (only for i=1)
            if i == 1:
                x_base = int((38 + 20 * i) * x_scale)
                y_center = y_dim / 2
                amplitude_curve = 120
                colony_y_positions = np.linspace(0, y_dim - 1, num_colonies_per_line * 2)
                for colony_y in colony_y_positions:
                    y_normalized = np.clip((colony_y - y_center) / (y_dim / 2), -1, 1)
                    curve_offset = amplitude_curve * (1 - y_normalized**2)
                    curve_x = x_base - curve_offset
                    center_x = int(np.clip(curve_x + np.random.uniform(-8, 8), 0, x_dim - 1))
                    center_y = int(np.clip(colony_y + np.random.uniform(-5, 5), 0, y_dim - 1))
                    radius = np.random.uniform(10, max_radius)
                    cloud_points = create_random_cloud(center_x, center_y, z, radius, z_dim, y_dim, x_dim)
                    for x, y, z_val in cloud_points:
                        if z_val == z:
                            random_value = np.random.randint(6, 11)
                            data[0, random_value, z, y, x] = 1
            
            # Channel 1: Semi-sinusoidal horizontal
            y_base = int((18 + 20 * i) * y_scale)
            target_peak_y = 150
            amplitude = max(50, min(target_peak_y - y_base, y_dim - y_base - 50))
            frequency = 2 * np.pi / (x_dim / 3)
            phase = i * np.pi / 2
            x_samples = []
            k_range = int(x_dim * frequency / (2 * np.pi)) + 1
            for k in range(-1, k_range + 1):
                peak_x = (np.pi / 2 + 2 * np.pi * k - phase) / frequency
                if 0 <= peak_x < x_dim:
                    x_samples.append(peak_x)
            if len(x_samples) > num_colonies_per_line * 4:
                indices = np.linspace(0, len(x_samples) - 1, num_colonies_per_line * 2, dtype=int)
                x_samples = [x_samples[idx] for idx in indices]
            for x_sample in x_samples:
                peak_y = y_base + amplitude
                center_x = int(np.clip(x_sample + np.random.uniform(-8, 8), 0, x_dim - 1))
                center_y = int(np.clip(peak_y + np.random.uniform(-5, 5), 0, y_dim - 1))
                radius = np.random.uniform(10, max_radius)
                cloud_points = create_random_cloud(center_x, center_y, z, radius, z_dim, y_dim, x_dim)
                for x, y, z_val in cloud_points:
                    if z_val == z:
                        random_value = np.random.randint(6, 11)
                        data[1, random_value, z, y, x] = 1
            
            # Channel 3: Diagonal lines with slope -1
            d_values = [int(60 * y_scale), int(120 * y_scale)]
            for d in d_values:
                x_samples = np.linspace(max(0, d - y_dim + 1), min(x_dim - 1, d), num_colonies_per_line)
                for x_sample in x_samples:
                    if x_sample < 0 or x_sample >= x_dim:
                        continue
                    y_line = -x_sample + d
                    if y_line < 0 or y_line >= y_dim:
                        continue
                    perp_offset = np.random.uniform(-15, 15)
                    center_x = int(np.clip(x_sample + perp_offset / np.sqrt(2), 0, x_dim - 1))
                    center_y = int(np.clip(y_line + perp_offset / np.sqrt(2), 0, y_dim - 1))
                    radius = np.random.uniform(10, max_radius)
                    cloud_points = create_random_cloud(center_x, center_y, z, radius, z_dim, y_dim, x_dim)
                    for x, y, z_val in cloud_points:
                        if z_val == z:
                            random_value = np.random.randint(6, 11)
                            data[3, random_value, z, y, x] = 1
    return data

# ============================================================================
# TEXTURE 3: Linear Pattern
# ============================================================================
def generate_linear_texture():
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    for i in range(0, 3):
        for z in range(z_dim):
            # Channel 0: Vertical strips
            for y in range(y_dim):
                x_start = max(0, int((38+20*i) * x_scale))
                x_end = min(x_dim, int((43+20*i) * x_scale))
                for x in range(x_start, x_end):
                    random_value = np.random.randint(6, 11)
                    data[0, random_value, z, y, x] = 1
            # Channel 1: Horizontal strips
            y_start = max(0, int((18+20*i) * y_scale))
            y_end = min(y_dim, int((23+20*i) * y_scale))
            for y in range(y_start, y_end):
                for x in range(x_dim):
                    random_value = np.random.randint(6, 11)
                    data[1, random_value, z, y, x] = 1
            # Channel 2: Diagonal slope 1
            strip_width = int(3 * max(x_scale, y_scale))
            c_values = [int(-60 * x_scale), 0]
            for c in c_values:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y - x - c) <= strip_width:
                            random_value = np.random.randint(6, 11)
                            data[2, random_value, z, y, x] = 1
            # Channel 3: Diagonal slope -1
            d_values = [int(60 * y_scale), int(120 * y_scale)]
            for d in d_values:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y + x - d) <= strip_width:
                            random_value = np.random.randint(6, 11)
                            data[3, random_value, z, y, x] = 1
    return data

# ============================================================================
# TEXTURE 4: Olympic Rings Pattern
# ============================================================================
def generate_olympic_texture():
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    stripe_width = 30
    circles = [
        (0, 200, 150, 120),
        (1, 420, 150, 120),
        (2, 300, 250, 120),
        (3, 350, 330, 200),
    ]
    for channel, center_x, center_y, radius in circles:
        create_striped_circle(data, channel, center_x, center_y, radius, stripe_width, z_dim, y_dim, x_dim)
    return data

# ============================================================================
# TEXTURE 5: Oval Pattern
# ============================================================================
def generate_oval_texture():
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    base_center_x = x_dim // 2
    base_center_y = y_dim // 2
    radius_ch0_inner = 50
    radius_ch0_outer = 100
    radius_ch1_inner = 80
    radius_ch1_outer = 110
    cloud_radius = 20
    num_cloud_points = 5
    points_per_cloud = 100
    x_stretch = 1.5
    z_center = z_dim // 2
    z_spread = 10
    max_centroid_distance = 5
    
    for z in range(z_dim):
        # Channel 0: Oval strip
        for y in range(y_dim):
            for x in range(x_dim):
                dx = (x - base_center_x) / x_stretch
                dy = y - base_center_y
                elliptical_dist = np.sqrt(dx**2 + dy**2)
                if radius_ch0_inner <= elliptical_dist <= radius_ch0_outer:
                    random_value = np.random.randint(6, 11)
                    data[0, random_value, z, y, x] = 1
        # Channel 1: Oval strip
        for y in range(y_dim):
            for x in range(x_dim):
                dx = (x - base_center_x) / x_stretch
                dy = y - base_center_y
                elliptical_dist = np.sqrt(dx**2 + dy**2)
                if radius_ch1_inner <= elliptical_dist <= radius_ch1_outer:
                    random_value = np.random.randint(6, 11)
                    data[1, random_value, z, y, x] = 1
    
    # Channel 2 and 3: 3D cloud points
    ch2_centroids = []
    for _ in range(num_cloud_points):
        cloud_angle = np.random.uniform(0, 2 * np.pi)
        cloud_radial_distance = np.random.uniform(radius_ch1_inner+5, radius_ch1_inner-5)
        cloud_center_x = int(np.clip(base_center_x + cloud_radial_distance * x_stretch * np.cos(cloud_angle), cloud_radius, x_dim - cloud_radius - 1))
        cloud_center_y = int(np.clip(base_center_y + cloud_radial_distance * np.sin(cloud_angle), cloud_radius, y_dim - cloud_radius - 1))
        ch2_centroids.append((cloud_center_x, cloud_center_y))
        cloud_points = create_concentrated_3d_cloud_point(cloud_center_x, cloud_center_y, z_center, cloud_radius, z_spread, points_per_cloud, z_dim, x_dim, y_dim)
        for x, y, z_val in cloud_points:
            if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z_val < z_dim:
                random_value = np.random.randint(6, 11)
                data[2, random_value, z_val, y, x] = 1
    
    for ch2_x, ch2_y in ch2_centroids:
        offset_angle = np.random.uniform(0, 2 * np.pi)
        offset_distance = np.random.uniform(0, max_centroid_distance)
        ch3_center_x = int(np.clip(ch2_x + offset_distance * np.cos(offset_angle), cloud_radius, x_dim - cloud_radius - 1))
        ch3_center_y = int(np.clip(ch2_y + offset_distance * np.sin(offset_angle), cloud_radius, y_dim - cloud_radius - 1))
        cloud_points = create_concentrated_3d_cloud_point(ch3_center_x, ch3_center_y, z_center, cloud_radius, z_spread, points_per_cloud, z_dim, x_dim, y_dim)
        for x, y, z_val in cloud_points:
            if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z_val < z_dim:
                random_value = np.random.randint(6, 11)
                data[3, random_value, z_val, y, x] = 1
    return data

# ============================================================================
# GENERATE ALL TEXTURES
# ============================================================================
print("Generating all textures...")

print("  1/5 - Generating Sinusoid texture...")
sinusoid_data = generate_sinusoid_texture()
np.save('groundtruth_sinus.npy', sinusoid_data)
print("       ✓ Saved: groundtruth_sinus.npy")

print("  2/5 - Generating Colonies texture...")
colonies_data = generate_colonies_texture()
np.save('groundtruth_colonies.npy', colonies_data)
print("       ✓ Saved: groundtruth_colonies.npy")

print("  3/5 - Generating Linear texture...")
linear_data = generate_linear_texture()
np.save('groundtruth.npy', linear_data)
print("       ✓ Saved: groundtruth.npy")

print("  4/5 - Generating Olympic Rings texture...")
olympic_data = generate_olympic_texture()
np.save('groundtruth_circles.npy', olympic_data)
print("       ✓ Saved: groundtruth_circles.npy")

print("  5/5 - Generating Oval texture...")
oval_data = generate_oval_texture()
np.save('oval_stripes.npy', oval_data)
print("       ✓ Saved: oval_stripes.npy")

print("\n" + "="*60)
print("ALL TEXTURES GENERATED SUCCESSFULLY!")
print("="*60)
print(f"Data shape: {sinusoid_data.shape}")
print(f"  - Channels: {num_channels}")
print(f"  - Values: {num_values}")
print(f"  - Z dimension: {z_dim}")
print(f"  - Y dimension: {y_dim}")
print(f"  - X dimension: {x_dim}")
print("\nGenerated files:")
print("  1. groundtruth_sinus.npy   - Sinusoidal patterns")
print("  2. groundtruth_colonies.npy - Colony/cloud patterns")
print("  3. groundtruth.npy         - Linear stripe patterns")
print("  4. groundtruth_circles.npy - Olympic ring patterns")
print("  5. oval_stripes.npy        - Oval/ellipse patterns")

Run all textures features 

In [ ]:
"""
Complete Scale Experiment Script for ALL TEXTURES
Creates ground truth for each texture, trains model, extracts coordinates, 
and computes accuracy (radius-aware IoU) for all textures and scales.

IMPORTANT: Texture generation functions are IDENTICAL to Cell 1.
Only dimensions are scaled.

At the end, produces an AVERAGE TABLE summarizing results across all textures and scales.
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.data import Data, Batch
import os
import time
import pandas as pd
from tqdm import tqdm
import pickle
import random

# ============================================================================
# Configuration
# ============================================================================
# For Google Colab, change this to your Google Drive path
# BASE_DIR = '/content/drive/MyDrive/SSGAT/Scale'
BASE_DIR = './Scale_Results'
os.makedirs(BASE_DIR, exist_ok=True)

# Original dimensions (same as Cell 1)
ORIGINAL_X = 681
ORIGINAL_Y = 344
ORIGINAL_Z = 49

# All scales to test
SCALES = [
    ('original', 1, 1, 1),
    ('2x1y1z', 2, 1, 1),
    ('1x2y1z', 1, 2, 1),
    ('1x1y2z', 1, 1, 2),
    ('2x2y1z', 2, 2, 1),
    ('1x2y2z', 1, 2, 2),
    ('2x2y2z', 2, 2, 2),
    ('3x1y1z', 3, 1, 1),
    ('3x2y1z', 3, 2, 1),
    ('3x1y2z', 3, 1, 2),
    ('3x2y2z', 3, 2, 2)
]
# All texture types to test
TEXTURES = ['sinusoid', 'colonies', 'linear', 'olympic', 'oval']

TOP_K = 100
Z_FIXED = 5
MAX_RADIUS = 10
STEP_SIZE = 5


# ============================================================================
# Helper Functions
# ============================================================================
def calculate_distance(x1, y1, z1, x2, y2, z2):
    """Calculate 3D Euclidean distance between two points"""
    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)


# ============================================================================
# TEXTURE GENERATION FUNCTIONS - IDENTICAL TO CELL 1
# ============================================================================

def create_sine_strip(data, channel, y_center, strip_size, amplitude, period, z_dim, y_dim, x_dim):
    """Create a sine wave strip pattern in x direction."""
    strip_half = strip_size // 2
    for z in range(z_dim):
        for x in range(x_dim):
            y_sine = y_center + amplitude * np.sin(2 * np.pi * x / period)
            y_sine_int = int(round(y_sine))
            y_start = max(0, y_sine_int - strip_half)
            y_end = min(y_dim, y_sine_int + strip_half + 1)
            for y in range(y_start, y_end):
                random_value = np.random.randint(0, 11)
                data[channel, random_value, z, y, x] = 1

def create_line_strip(data, channel, y_center, strip_size, z_dim, y_dim, x_dim):
    """Create a straight line strip in x direction at y = y_center."""
    strip_half = strip_size // 2
    y_start = max(0, y_center - strip_half)
    y_end = min(y_dim, y_center + strip_half + 1)
    for z in range(z_dim):
        for x in range(x_dim):
            for y in range(y_start, y_end):
                random_value = np.random.randint(0, 11)
                data[channel, random_value, z, y, x] = 1

def create_random_cloud(center_x, center_y, center_z, radius, z_dim, y_dim, x_dim):
    """Create a random cloud of points around a center with random shape"""
    num_points = np.random.randint(20, 101)
    shape_type = np.random.choice(['sphere', 'ellipsoid', 'elongated'])
    points = []
    for _ in range(num_points):
        if shape_type == 'sphere':
            theta = np.random.uniform(0, 2 * np.pi)
            phi = np.random.uniform(0, np.pi)
            r = np.random.uniform(0, radius) ** (1/3)
            dx = r * np.sin(phi) * np.cos(theta)
            dy = r * np.sin(phi) * np.sin(theta)
            dz = r * np.cos(phi)
        elif shape_type == 'ellipsoid':
            axes = np.random.uniform(0.3, 1.0, 3) * radius
            theta = np.random.uniform(0, 2 * np.pi)
            phi = np.random.uniform(0, np.pi)
            r = np.random.uniform(0, 1) ** (1/3)
            dx = r * axes[0] * np.sin(phi) * np.cos(theta)
            dy = r * axes[1] * np.sin(phi) * np.sin(theta)
            dz = r * axes[2] * np.cos(phi)
        else:  # elongated
            direction = np.random.uniform(-1, 1, 3)
            direction = direction / np.linalg.norm(direction)
            r_parallel = np.random.uniform(0, radius)
            r_perp = np.random.uniform(0, radius * 0.4)
            perp1 = np.cross(direction, [1, 0, 0])
            if np.linalg.norm(perp1) < 0.1:
                perp1 = np.cross(direction, [0, 1, 0])
            perp1 = perp1 / np.linalg.norm(perp1)
            perp2 = np.cross(direction, perp1)
            perp2 = perp2 / np.linalg.norm(perp2)
            angle = np.random.uniform(0, 2 * np.pi)
            offset = r_parallel * direction + r_perp * (np.cos(angle) * perp1 + np.sin(angle) * perp2)
            dx, dy, dz = offset[0], offset[1], offset[2]
        
        x = int(center_x + dx)
        y = int(center_y + dy)
        z = int(center_z + dz)
        if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z < z_dim:
            points.append((x, y, z))
    return points

def create_striped_circle(data, channel, center_x, center_y, max_radius, stripe_width, z_dim, y_dim, x_dim):
    """Create a circle with a single stripe/ring."""
    inner_radius = max_radius - stripe_width
    outer_radius = max_radius
    for z in range(z_dim):
        for y in range(y_dim):
            for x in range(x_dim):
                dx = x - center_x
                dy = y - center_y
                distance = np.sqrt(dx**2 + dy**2)
                if inner_radius <= distance <= outer_radius:
                    random_value = np.random.randint(0, 11)
                    data[channel, random_value, z, y, x] = 1

def create_concentrated_3d_cloud_point(center_x, center_y, center_z, radius_xy, radius_z, num_points, z_dim, x_dim, y_dim):
    """Create a concentrated 3D cloud of points around a center"""
    cloud_points = []
    for _ in range(num_points):
        angle = np.random.uniform(0, 2 * np.pi)
        r_xy = radius_xy * np.sqrt(np.random.uniform(0, 1))
        x = int(center_x + r_xy * np.cos(angle))
        y = int(center_y + r_xy * np.sin(angle))
        dz = int(np.random.uniform(-radius_z, radius_z))
        z = center_z + dz
        z = int(np.clip(z, 0, z_dim - 1))
        if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z < z_dim:
            cloud_points.append((x, y, z))
    return cloud_points


# ============================================================================
# TEXTURE 1: Sinusoidal Pattern - SAME AS CELL 1
# ============================================================================
def generate_sinusoid_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Generate sinusoidal texture - IDENTICAL to Cell 1"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    y_center = 100
    strip_size = 20
    sine_params = [(0, 100, 400), (1, 50, 200), (2, 150, 300)]
    for channel, amplitude, period in sine_params:
        create_sine_strip(data, channel, y_center, strip_size, amplitude, period, z_dim, y_dim, x_dim)
    create_line_strip(data, 3, y_center, strip_size, z_dim, y_dim, x_dim)
    return data


# ============================================================================
# TEXTURE 2: Colonies Pattern - SAME AS CELL 1
# ============================================================================
def generate_colonies_texture(num_channels, num_values, z_dim, y_dim, x_dim, x_scale, y_scale):
    """Generate colonies texture - IDENTICAL to Cell 1"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    max_radius = 50
    num_colonies_per_line = 8
    
    # Channel 0: 3D Arc
    num_arc_points = num_colonies_per_line * 3
    t_values = np.linspace(0, 1, num_arc_points)
    arc_amplitude = 0.3
    
    for t in t_values:
        x_arc_base = t * (x_dim - 1)
        y_arc_base = t * (y_dim - 1)
        z_arc_base = t * (z_dim - 1)
        angle = t * np.pi
        arc_elevation_y = arc_amplitude * (y_dim - 1) * np.sin(angle)
        arc_elevation_z = arc_amplitude * (z_dim - 1) * np.sin(angle)
        x_arc = int(np.clip(x_arc_base, 0, x_dim - 1))
        y_arc = int(np.clip(y_arc_base + arc_elevation_y, 0, y_dim - 1))
        z_arc = int(np.clip(z_arc_base + arc_elevation_z, 0, z_dim - 1))
        radius = np.random.uniform(10, max_radius)
        cloud_points = create_random_cloud(x_arc, y_arc, z_arc, radius, z_dim, y_dim, x_dim)
        for x, y, z_val in cloud_points:
            if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z_val < z_dim:
                random_value = np.random.randint(6, 11)
                data[0, random_value, z_val, y, x] = 1
    
    # Generate remaining channels
    for i in range(0, 3):
        for z in range(z_dim):
            # Channel 0: Concave curve (only for i=1)
            if i == 1:
                x_base = int((38 + 20 * i) * x_scale)
                y_center = y_dim / 2
                amplitude_curve = 120
                colony_y_positions = np.linspace(0, y_dim - 1, num_colonies_per_line * 2)
                for colony_y in colony_y_positions:
                    y_normalized = np.clip((colony_y - y_center) / (y_dim / 2), -1, 1)
                    curve_offset = amplitude_curve * (1 - y_normalized**2)
                    curve_x = x_base - curve_offset
                    center_x = int(np.clip(curve_x + np.random.uniform(-8, 8), 0, x_dim - 1))
                    center_y = int(np.clip(colony_y + np.random.uniform(-5, 5), 0, y_dim - 1))
                    radius = np.random.uniform(10, max_radius)
                    cloud_points = create_random_cloud(center_x, center_y, z, radius, z_dim, y_dim, x_dim)
                    for x, y, z_val in cloud_points:
                        if z_val == z:
                            random_value = np.random.randint(6, 11)
                            data[0, random_value, z, y, x] = 1
            
            # Channel 1: Semi-sinusoidal horizontal
            y_base = int((18 + 20 * i) * y_scale)
            target_peak_y = 150
            amplitude = max(50, min(target_peak_y - y_base, y_dim - y_base - 50))
            frequency = 2 * np.pi / (x_dim / 3)
            phase = i * np.pi / 2
            x_samples = []
            k_range = int(x_dim * frequency / (2 * np.pi)) + 1
            for k in range(-1, k_range + 1):
                peak_x = (np.pi / 2 + 2 * np.pi * k - phase) / frequency
                if 0 <= peak_x < x_dim:
                    x_samples.append(peak_x)
            if len(x_samples) > num_colonies_per_line * 4:
                indices = np.linspace(0, len(x_samples) - 1, num_colonies_per_line * 2, dtype=int)
                x_samples = [x_samples[idx] for idx in indices]
            for x_sample in x_samples:
                peak_y = y_base + amplitude
                center_x = int(np.clip(x_sample + np.random.uniform(-8, 8), 0, x_dim - 1))
                center_y = int(np.clip(peak_y + np.random.uniform(-5, 5), 0, y_dim - 1))
                radius = np.random.uniform(10, max_radius)
                cloud_points = create_random_cloud(center_x, center_y, z, radius, z_dim, y_dim, x_dim)
                for x, y, z_val in cloud_points:
                    if z_val == z:
                        random_value = np.random.randint(6, 11)
                        data[1, random_value, z, y, x] = 1
            
            # Channel 3: Diagonal lines with slope -1
            d_values = [int(60 * y_scale), int(120 * y_scale)]
            for d in d_values:
                x_samples_diag = np.linspace(max(0, d - y_dim + 1), min(x_dim - 1, d), num_colonies_per_line)
                for x_sample in x_samples_diag:
                    if x_sample < 0 or x_sample >= x_dim:
                        continue
                    y_line = -x_sample + d
                    if y_line < 0 or y_line >= y_dim:
                        continue
                    perp_offset = np.random.uniform(-15, 15)
                    center_x = int(np.clip(x_sample + perp_offset / np.sqrt(2), 0, x_dim - 1))
                    center_y = int(np.clip(y_line + perp_offset / np.sqrt(2), 0, y_dim - 1))
                    radius = np.random.uniform(10, max_radius)
                    cloud_points = create_random_cloud(center_x, center_y, z, radius, z_dim, y_dim, x_dim)
                    for x, y, z_val in cloud_points:
                        if z_val == z:
                            random_value = np.random.randint(6, 11)
                            data[3, random_value, z, y, x] = 1
    return data


# ============================================================================
# TEXTURE 3: Linear Pattern - SAME AS CELL 1
# ============================================================================
def generate_linear_texture(num_channels, num_values, z_dim, y_dim, x_dim, x_scale, y_scale):
    """Generate linear texture - IDENTICAL to Cell 1"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    for i in range(0, 3):
        for z in range(z_dim):
            # Channel 0: Vertical strips
            for y in range(y_dim):
                x_start = max(0, int((38+20*i) * x_scale))
                x_end = min(x_dim, int((43+20*i) * x_scale))
                for x in range(x_start, x_end):
                    random_value = np.random.randint(6, 11)
                    data[0, random_value, z, y, x] = 1
            # Channel 1: Horizontal strips
            y_start = max(0, int((18+20*i) * y_scale))
            y_end = min(y_dim, int((23+20*i) * y_scale))
            for y in range(y_start, y_end):
                for x in range(x_dim):
                    random_value = np.random.randint(6, 11)
                    data[1, random_value, z, y, x] = 1
            # Channel 2: Diagonal slope 1
            strip_width = int(3 * max(x_scale, y_scale))
            c_values = [int(-60 * x_scale), 0]
            for c in c_values:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y - x - c) <= strip_width:
                            random_value = np.random.randint(6, 11)
                            data[2, random_value, z, y, x] = 1
            # Channel 3: Diagonal slope -1
            d_values = [int(60 * y_scale), int(120 * y_scale)]
            for d in d_values:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y + x - d) <= strip_width:
                            random_value = np.random.randint(6, 11)
                            data[3, random_value, z, y, x] = 1
    return data


# ============================================================================
# TEXTURE 4: Olympic Rings Pattern - SAME AS CELL 1
# ============================================================================
def generate_olympic_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Generate olympic rings texture - IDENTICAL to Cell 1"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    stripe_width = 30
    circles = [
        (0, 200, 150, 120),
        (1, 420, 150, 120),
        (2, 300, 250, 120),
        (3, 350, 330, 200),
    ]
    for channel, center_x, center_y, radius in circles:
        create_striped_circle(data, channel, center_x, center_y, radius, stripe_width, z_dim, y_dim, x_dim)
    return data


# ============================================================================
# TEXTURE 5: Oval Pattern - SAME AS CELL 1
# ============================================================================
def generate_oval_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Generate oval texture - IDENTICAL to Cell 1"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    base_center_x = x_dim // 2
    base_center_y = y_dim // 2
    radius_ch0_inner = 50
    radius_ch0_outer = 100
    radius_ch1_inner = 80
    radius_ch1_outer = 110
    cloud_radius = 20
    num_cloud_points = 5
    points_per_cloud = 100
    x_stretch = 1.5
    z_center = z_dim // 2
    z_spread = 10
    max_centroid_distance = 5
    
    for z in range(z_dim):
        # Channel 0: Oval strip
        for y in range(y_dim):
            for x in range(x_dim):
                dx = (x - base_center_x) / x_stretch
                dy = y - base_center_y
                elliptical_dist = np.sqrt(dx**2 + dy**2)
                if radius_ch0_inner <= elliptical_dist <= radius_ch0_outer:
                    random_value = np.random.randint(6, 11)
                    data[0, random_value, z, y, x] = 1
        # Channel 1: Oval strip
        for y in range(y_dim):
            for x in range(x_dim):
                dx = (x - base_center_x) / x_stretch
                dy = y - base_center_y
                elliptical_dist = np.sqrt(dx**2 + dy**2)
                if radius_ch1_inner <= elliptical_dist <= radius_ch1_outer:
                    random_value = np.random.randint(6, 11)
                    data[1, random_value, z, y, x] = 1
    
    # Channel 2 and 3: 3D cloud points
    ch2_centroids = []
    for _ in range(num_cloud_points):
        cloud_angle = np.random.uniform(0, 2 * np.pi)
        cloud_radial_distance = np.random.uniform(radius_ch1_inner+5, radius_ch1_inner-5)
        cloud_center_x = int(np.clip(base_center_x + cloud_radial_distance * x_stretch * np.cos(cloud_angle), cloud_radius, x_dim - cloud_radius - 1))
        cloud_center_y = int(np.clip(base_center_y + cloud_radial_distance * np.sin(cloud_angle), cloud_radius, y_dim - cloud_radius - 1))
        ch2_centroids.append((cloud_center_x, cloud_center_y))
        cloud_points = create_concentrated_3d_cloud_point(cloud_center_x, cloud_center_y, z_center, cloud_radius, z_spread, points_per_cloud, z_dim, x_dim, y_dim)
        for x, y, z_val in cloud_points:
            if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z_val < z_dim:
                random_value = np.random.randint(6, 11)
                data[2, random_value, z_val, y, x] = 1
    
    for ch2_x, ch2_y in ch2_centroids:
        offset_angle = np.random.uniform(0, 2 * np.pi)
        offset_distance = np.random.uniform(0, max_centroid_distance)
        ch3_center_x = int(np.clip(ch2_x + offset_distance * np.cos(offset_angle), cloud_radius, x_dim - cloud_radius - 1))
        ch3_center_y = int(np.clip(ch2_y + offset_distance * np.sin(offset_angle), cloud_radius, y_dim - cloud_radius - 1))
        cloud_points = create_concentrated_3d_cloud_point(ch3_center_x, ch3_center_y, z_center, cloud_radius, z_spread, points_per_cloud, z_dim, x_dim, y_dim)
        for x, y, z_val in cloud_points:
            if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z_val < z_dim:
                random_value = np.random.randint(6, 11)
                data[3, random_value, z_val, y, x] = 1
    return data


# ============================================================================
# 1. Create Ground Truth with Different Scales for ALL TEXTURES
# Using the EXACT SAME texture generation functions defined above (same as Cell 1)
# ============================================================================

def create_groundtruth(texture_type, scale_name, x_scale_factor, y_scale_factor, z_scale_factor, output_dir):
    """
    Create ground truth data with specified texture type and scale.
    
    Uses the SAME texture generation functions as Cell 1.
    Dimensions are scaled by the scale factors.
    
    Args:
        texture_type: 'sinusoid', 'colonies', 'linear', 'olympic', 'oval'
        scale_name: Name for this scale configuration (e.g., 'original', '2x1y1z')
        x_scale_factor, y_scale_factor, z_scale_factor: Scale multipliers for dimensions
        output_dir: Directory to save the ground truth file
    """
    print(f"\n{'='*70}")
    print(f"Creating Ground Truth: {texture_type} - {scale_name} (X={x_scale_factor}x, Y={y_scale_factor}x, Z={z_scale_factor}x)")
    print(f"{'='*70}")

    num_channels = 4
    value_range = (0, 10)
    num_values = value_range[1] - value_range[0] + 1

    # Calculate scaled dimensions
    x_dim = ORIGINAL_X * x_scale_factor
    y_dim = ORIGINAL_Y * y_scale_factor
    z_dim = ORIGINAL_Z * z_scale_factor
    
    # Calculate local scaling factors (same logic as Cell 1: x_scale = x_dim / 172, y_scale = y_dim / 87)
    local_x_scale = x_dim / 172
    local_y_scale = y_dim / 87

    print(f"  Dimensions: X={x_dim}, Y={y_dim}, Z={z_dim}")

    # Generate texture using the SAME functions defined above (IDENTICAL to Cell 1)
    # These functions are: generate_sinusoid_texture, generate_colonies_texture, etc.
    if texture_type == 'sinusoid':
        data = generate_sinusoid_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    elif texture_type == 'colonies':
        data = generate_colonies_texture(num_channels, num_values, z_dim, y_dim, x_dim, local_x_scale, local_y_scale)
    elif texture_type == 'linear':
        data = generate_linear_texture(num_channels, num_values, z_dim, y_dim, x_dim, local_x_scale, local_y_scale)
    elif texture_type == 'olympic':
        data = generate_olympic_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    elif texture_type == 'oval':
        data = generate_oval_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    else:
        raise ValueError(f"Unknown texture type: {texture_type}")

    # Save ground truth
    filename = f'groundtruth_{texture_type}_{scale_name}.npy'
    filepath = os.path.join(output_dir, filename)
    np.save(filepath, data)
    print(f"✓ Saved ground truth to: {filepath}")
    print(f"  Shape: {data.shape}, Size: {data.nbytes / 1024 / 1024:.2f} MB")

    return data, filepath


# ============================================================================
# 2. Create Subgraphs from Ground Truth
# ============================================================================
def create_subgraphs(data, texture_type, scale_name, output_dir):
    """Create subgraphs from ground truth data"""
    print(f"\n{'='*70}")
    print(f"Creating Subgraphs: {texture_type} - {scale_name}")
    print(f"{'='*70}")

    num_channels, num_values, z_dim, y_dim, x_dim = data.shape
    z_idx = Z_FIXED if Z_FIXED < z_dim else z_dim // 2

    print(f"  Using z={z_idx}, Dimensions: {y_dim}x{x_dim}")

    # Pre-compute intensity matrix and mask per channel
    print("  Pre-computing intensity matrix...")
    intensity_matrix = np.zeros((y_dim, x_dim, num_channels), dtype=np.float32)
    channel_mask = np.zeros((y_dim, x_dim, num_channels), dtype=bool)

    for channel in range(num_channels):
        # channel_data: (V, Y, X) at fixed z
        channel_data = data[channel, :, z_idx, :, :]
        value_indices = np.argmax(channel_data, axis=0)
        mask = channel_data.sum(axis=0) > 0
        intensity_matrix[:, :, channel] = np.where(mask, value_indices.astype(np.float32), 0.0)
        channel_mask[:, :, channel] = mask

    channel_counts = channel_mask.sum(axis=2)

    # Generate graph centers
    graph_centers = []
    for x in range(0, x_dim, STEP_SIZE):
        for y in range(0, y_dim, STEP_SIZE):
            graph_centers.append((x, y, z_idx))

    print(f"  Total graph centers: {len(graph_centers)}")

    # Create subgraphs
    all_subgraphs = []
    print("  Creating subgraphs...")

    for center_idx, (center_x, center_y, center_z) in enumerate(tqdm(graph_centers, desc="Processing")):
        x_min = max(0, int(center_x - MAX_RADIUS))
        x_max = min(x_dim, int(center_x + MAX_RADIUS) + 1)
        y_min = max(0, int(center_y - MAX_RADIUS))
        y_max = min(y_dim, int(center_y + MAX_RADIUS) + 1)

        nodes = []
        node_positions = []
        node_active_channels = []

        for y in range(y_min, y_max):
            for x in range(x_min, x_max):
                if channel_counts[y, x] > 0:
                    distance = calculate_distance(center_x, center_y, center_z, x, y, z_idx)
                    if distance <= MAX_RADIUS:
                        active_channels = np.where(channel_mask[y, x, :])[0].tolist()
                        intensities = intensity_matrix[y, x, active_channels]
                        nodes.append(intensities)
                        node_positions.append((x, y, z_idx))
                        node_active_channels.append(active_channels)

        if len(nodes) == 0:
            continue

        max_channels = max(len(channels) for channels in node_active_channels)
        padded_nodes = []
        for i, intensities in enumerate(nodes):
            active_ch = node_active_channels[i]
            num_ch = len(active_ch)
            if num_ch < max_channels:
                padded = np.zeros(max_channels, dtype=np.float32)
                padded[:num_ch] = intensities
                padded_nodes.append(padded)
            else:
                padded_nodes.append(intensities)

        node_features = np.array(padded_nodes, dtype=np.float32)
        node_positions_array = np.array(node_positions, dtype=np.int32)
        num_nodes = len(node_features)

        if num_nodes < 2:
            continue

        # Create edges
        node_positions_np = node_positions_array.astype(np.float32)
        if num_nodes < 50000:
            diff = node_positions_np[:, np.newaxis, :] - node_positions_np[np.newaxis, :, :]
            distances_matrix = np.sqrt(np.sum(diff**2, axis=2))
            edge_mask = (distances_matrix <= MAX_RADIUS) & (distances_matrix > 0)
            edge_i, edge_j = np.where(edge_mask)
        else:
            # For large graphs, use iterative approach
            edge_i, edge_j = [], []
            for i in range(num_nodes):
                for j in range(i+1, num_nodes):
                    dist = calculate_distance(
                        node_positions_array[i,0], node_positions_array[i,1], node_positions_array[i,2],
                        node_positions_array[j,0], node_positions_array[j,1], node_positions_array[j,2]
                    )
                    if 0 < dist <= MAX_RADIUS:
                        edge_i.extend([i, j])
                        edge_j.extend([j, i])
            edge_i, edge_j = np.array(edge_i), np.array(edge_j)

        if len(edge_i) == 0:
            continue

        edge_index = torch.tensor([edge_i, edge_j], dtype=torch.long)
        edge_weights_np = np.array([
            calculate_distance(
                node_positions_array[edge_i[k],0], node_positions_array[edge_i[k],1], node_positions_array[edge_i[k],2],
                node_positions_array[edge_j[k],0], node_positions_array[edge_j[k],1], node_positions_array[edge_j[k],2]
            ) for k in range(len(edge_i))
        ], dtype=np.float32)

        node_values = node_features.sum(axis=1)

        edge_attr = torch.tensor(edge_weights_np, dtype=torch.float32)
        x_tensor = torch.tensor(node_features, dtype=torch.float32)
        node_values_tensor = torch.tensor(node_values, dtype=torch.float32)

        graph = Data(
            x=x_tensor,
            edge_index=edge_index,
            edge_attr=edge_attr,
            center=(center_x, center_y, center_z),
            center_idx=center_idx,
            node_positions=[tuple(pos) for pos in node_positions_array],
            node_values=node_values_tensor,
            max_channels=max_channels
        )

        all_subgraphs.append(graph)

    # Save subgraphs
    filename = f'Subgraph_{texture_type}_{scale_name}.pt'
    filepath = os.path.join(output_dir, filename)
    torch.save(all_subgraphs, filepath)
    print(f"✓ Saved {len(all_subgraphs)} subgraphs to: {filepath}")
    print(f"  File size: {os.path.getsize(filepath) / 1024 / 1024:.2f} MB")

    return all_subgraphs, filepath


# ============================================================================
# 3. Model Classes and Functions
# ============================================================================

def prepare_graph_for_batching(graph, target_channels=4):
    """Prepare graph for batching by padding to target_channels"""
    x = graph.x.clone()
    current_channels = x.shape[1]
    if current_channels < target_channels:
        padding = torch.zeros(x.shape[0], target_channels - current_channels, dtype=x.dtype, device=x.device)
        x = torch.cat([x, padding], dim=1)
    elif current_channels > target_channels:
        x = x[:, :target_channels]

    clean_graph = Data(x=x, edge_index=graph.edge_index.clone())
    if hasattr(graph, 'edge_attr') and graph.edge_attr is not None:
        clean_graph.edge_attr = graph.edge_attr.clone()
    if hasattr(graph, 'center'):
        clean_graph._original_center = graph.center

    return clean_graph


def graph_augment(graph, node_mask_ratio=0.1, edge_drop_ratio=0.05, target_channels=4):
    """Augment graph for contrastive learning"""
    aug_graph = prepare_graph_for_batching(graph, target_channels=target_channels)

    # Keep gt_score on augmented graph if present
    if hasattr(graph, 'gt_score'):
        aug_graph.gt_score = graph.gt_score

    num_nodes = aug_graph.x.shape[0]
    num_mask = int(num_nodes * node_mask_ratio)
    if num_mask > 0:
        mask_indices = torch.randperm(num_nodes)[:num_mask]
        aug_graph.x[mask_indices] = 0.0

    if edge_drop_ratio > 0 and aug_graph.edge_index.shape[1] > 0:
        num_edges = aug_graph.edge_index.shape[1]
        num_drop = int(num_edges * edge_drop_ratio)
        if num_drop > 0:
            keep_indices = torch.randperm(num_edges)[:(num_edges - num_drop)]
            aug_graph.edge_index = aug_graph.edge_index[:, keep_indices]
            if hasattr(aug_graph, 'edge_attr') and aug_graph.edge_attr is not None:
                aug_graph.edge_attr = aug_graph.edge_attr[keep_indices]

    return aug_graph


class ContrastiveGAT(nn.Module):
    """Graph Attention Network with Self-Supervised Contrastive Learning"""
    def __init__(self, in_channels=4, hidden_channels=64, projection_dim=32, num_heads=4, dropout=0.1, edge_dim=None):
        super(ContrastiveGAT, self).__init__()
        self.edge_dim = edge_dim
        gat_kwargs = dict(dropout=dropout)
        if self.edge_dim is not None and self.edge_dim > 0:
            gat_kwargs["edge_dim"] = self.edge_dim

        self.gat1 = GATConv(in_channels=in_channels, out_channels=hidden_channels, heads=num_heads, concat=True, **gat_kwargs)
        self.gat2 = GATConv(in_channels=hidden_channels * num_heads, out_channels=hidden_channels, heads=num_heads, concat=True, **gat_kwargs)
        self.gat3 = GATConv(in_channels=hidden_channels * num_heads, out_channels=hidden_channels, heads=1, concat=False, **gat_kwargs)

        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(hidden_channels * num_heads)
        self.norm2 = nn.LayerNorm(hidden_channels * num_heads)
        self.pool_dim = hidden_channels * 3

        self.projection = nn.Sequential(
            nn.Linear(self.pool_dim, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, projection_dim)
        )

        self.interaction_head = nn.Sequential(
            nn.Linear(self.pool_dim, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, 1)
        )

    def encode(self, x, edge_index, edge_attr=None, batch=None):
        if self.edge_dim is not None and self.edge_dim > 0 and edge_attr is not None:
            ea = edge_attr
        else:
            ea = None

        x = self.gat1(x, edge_index, ea)
        x = self.norm1(x)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.gat2(x, edge_index, ea)
        x = self.norm2(x)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.gat3(x, edge_index, ea)
        x = F.elu(x)

        return x

    def forward(self, x, edge_index, edge_attr=None, batch=None):
        node_emb = self.encode(x, edge_index, edge_attr, batch)

        if batch is None:
            batch = torch.zeros(node_emb.shape[0], dtype=torch.long, device=node_emb.device)

        mean_pool = global_mean_pool(node_emb, batch)
        max_pool = global_max_pool(node_emb, batch)
        sum_pool = global_add_pool(node_emb, batch)

        graph_emb = torch.cat([mean_pool, max_pool, sum_pool], dim=1)

        proj_emb = self.projection(graph_emb)
        proj_emb = F.normalize(proj_emb, dim=1)

        interaction_score = self.interaction_head(graph_emb)

        return proj_emb, interaction_score


def contrastive_loss(z1, z2, temperature=0.1):
    """Contrastive loss (InfoNCE) for self-supervised learning"""
    batch_size = z1.shape[0]
    device = z1.device

    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    sim_matrix = torch.matmul(z1, z2.T) / temperature
    labels = torch.arange(batch_size, device=device)

    loss = F.cross_entropy(sim_matrix, labels)
    loss_reverse = F.cross_entropy(sim_matrix.T, labels)

    return (loss + loss_reverse) / 2.0


def train_contrastive_model(model, graphs, device, epochs=10, batch_size=32, lr=0.01, gradient_accumulation_steps=4):
    """Train model with self-supervised contrastive learning"""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    losses = []
    target_channels = max([g.x.shape[1] for g in graphs]) if len(graphs) > 0 else 4

    print(f"\n{'='*60}")
    print(f"Training Contrastive GAT Model")
    print(f"{'='*60}")
    print(f"  Epochs: {epochs}, Batch size: {batch_size}, LR: {lr}")
    print(f"  Total graphs: {len(graphs)}, Target channels: {target_channels}")

    if device.type == 'cuda':
        torch.cuda.empty_cache()

    for epoch in range(epochs):
        epoch_losses = []
        optimizer.zero_grad()

        shuffled_graphs = graphs.copy()
        random.shuffle(shuffled_graphs)

        for batch_idx, i in enumerate(range(0, len(shuffled_graphs), batch_size)):
            batch_graphs = shuffled_graphs[i:i+batch_size]

            aug1_graphs = [graph_augment(g, target_channels=target_channels) for g in batch_graphs]
            aug2_graphs = [graph_augment(g, target_channels=target_channels) for g in batch_graphs]

            for g in aug1_graphs + aug2_graphs:
                g.x = g.x.to(device)
                g.edge_index = g.edge_index.to(device)
                if hasattr(g, 'edge_attr') and g.edge_attr is not None:
                    g.edge_attr = g.edge_attr.to(device)

            try:
                batch1 = Batch.from_data_list(aug1_graphs)
                batch2 = Batch.from_data_list(aug2_graphs)
            except Exception as e:
                print(f"Error creating batch: {e}")
                continue

            z1, pred1 = model(batch1.x, batch1.edge_index, getattr(batch1, "edge_attr", None), batch1.batch)
            z2, _ = model(batch2.x, batch2.edge_index, getattr(batch2, "edge_attr", None), batch2.batch)

            # Contrastive loss
            loss_contrast = contrastive_loss(z1, z2, temperature=0.1)

            # Supervised regression loss on gt_score (if available)
            loss_reg = torch.tensor(0.0, device=device)
            if hasattr(batch1, "gt_score"):
                try:
                    gt_scores = batch1.gt_score.to(device).float()
                    pred_scores = pred1.view(-1)
                    # Normalize scores for stability
                    if gt_scores.std() > 0:
                        gt_scores = (gt_scores - gt_scores.mean()) / (gt_scores.std() + 1e-8)
                    if pred_scores.std() > 0:
                        pred_scores = (pred_scores - pred_scores.mean()) / (pred_scores.std() + 1e-8)
                    loss_reg = F.mse_loss(pred_scores, gt_scores)
                except:
                    pass

            # Combined loss
            loss = (loss_contrast + 0.5 * loss_reg) / gradient_accumulation_steps
            loss.backward()

            del z1, z2, batch1, batch2, aug1_graphs, aug2_graphs
            if device.type == 'cuda':
                torch.cuda.empty_cache()

            if (batch_idx + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            epoch_losses.append(loss.item() * gradient_accumulation_steps)

        if len(epoch_losses) > 0 and (len(shuffled_graphs) // batch_size) % gradient_accumulation_steps != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        avg_loss = np.mean(epoch_losses) if len(epoch_losses) > 0 else 0.0
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.6f}")

    print(f"\n✓ Training complete!")
    return model, losses


# ============================================================================
# 4. Extract Coordinates using Model (predictions)
# ============================================================================
def find_max_interaction_positions(model, graphs, device, top_k=100):
    """
    Find positions with maximum interaction scores using the trained model.
    Uses the model's interaction_score, not any hand-crafted heuristic.
    """
    model.eval()
    all_scores = []

    target_channels = max([g.x.shape[1] for g in graphs]) if len(graphs) > 0 else 4

    with torch.no_grad():
        for graph in tqdm(graphs, desc="Processing graphs (model)"):
            # Prepare node features to have consistent channel dimension
            x = graph.x.clone()
            current_channels = x.shape[1]
            if current_channels < target_channels:
                padding = torch.zeros(x.shape[0], target_channels - current_channels, dtype=x.dtype, device=x.device)
                x = torch.cat([x, padding], dim=1)
            elif current_channels > target_channels:
                x = x[:, :target_channels]

            x = x.to(device)
            edge_index = graph.edge_index.to(device)
            if hasattr(graph, 'edge_attr') and graph.edge_attr is not None:
                edge_attr = graph.edge_attr.to(device)
            else:
                edge_attr = None

            # Forward pass: get model interaction score for this graph
            _, interaction_score = model(x, edge_index, edge_attr, batch=None)
            model_score = interaction_score.item()

            center = graph.center
            x_pos, y_pos, z_pos = center

            # Optional extra info for analysis
            num_nodes = graph.x.shape[0]
            num_edges = graph.edge_index.shape[1]
            num_channels = graph.x.shape[1] if hasattr(graph, 'x') else 0
            edge_density = num_edges / num_nodes if num_nodes > 0 else 0

            all_scores.append({
                'x': x_pos,
                'y': y_pos,
                'z': z_pos,
                'num_nodes': num_nodes,
                'num_edges': num_edges,
                'num_channels': num_channels,
                'edge_density': edge_density,
                'model_score': model_score
            })

    if len(all_scores) > 0:
        # Sort by model_score (descending): higher score = more important
        all_scores.sort(key=lambda x: x['model_score'], reverse=True)
        top_positions = all_scores[:top_k]
        return top_positions

    return []


# ============================================================================
# 5. Extract Ground-Truth Coordinates from Data (Option A)
# ============================================================================
def attach_gt_scores_to_graphs(data, graphs):
    """
    Compute a ground-truth score per graph center (same definition
    as extract_gt_coordinates_from_data) and store it as graph.gt_score.
    """
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape

    # pattern_mask[z, y, x] = True if any channel / value > 0 at that voxel
    pattern_mask = (data > 0)          # (C, V, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=1)  # (C, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=0)  # (Z, Y, X)

    radius_sq = MAX_RADIUS ** 2

    for g in graphs:
        cx, cy, cz = g.center
        cx, cy, cz = int(cx), int(cy), int(cz)
        cz = max(0, min(z_dim - 1, cz))

        x_min = max(0, cx - MAX_RADIUS)
        x_max = min(x_dim, cx + MAX_RADIUS + 1)
        y_min = max(0, cy - MAX_RADIUS)
        y_max = min(y_dim, cy + MAX_RADIUS + 1)

        score = 0
        for y in range(y_min, y_max):
            dy2 = (y - cy) * (y - cy)
            for x in range(x_min, x_max):
                dx2 = (x - cx) * (x - cx)
                if dx2 + dy2 <= radius_sq:
                    if pattern_mask[cz, y, x]:
                        score += 1

        g.gt_score = float(score)

    return graphs


def extract_gt_coordinates_from_data(data, graphs, top_k=100):
    """
    Extract top coordinates from TRUE ground truth using Option A:
    A voxel is 'pattern' if ANY channel has a nonzero value.

    For each graph center, count how many pattern voxels are within radius MAX_RADIUS.
    Then pick the top_k centers with highest counts.
    """
    print("\nComputing ground-truth coordinates from data (Option A: ANY channel nonzero)...")

    num_channels, num_values, z_dim, y_dim, x_dim = data.shape

    # pattern_mask[z, y, x] = True if any channel / value > 0 at that voxel
    # data shape: (C, V, Z, Y, X)
    pattern_mask = (data > 0)        # bool (C, V, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=1)  # any over values -> (C, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=0)  # any over channels -> (Z, Y, X)

    all_scores = []
    radius_sq = MAX_RADIUS ** 2

    for graph in tqdm(graphs, desc="Processing graphs (GT)"):
        cx, cy, cz = graph.center
        cx, cy, cz = int(cx), int(cy), int(cz)

        # Safety clamp
        cz = max(0, min(z_dim - 1, cz))

        x_min = max(0, cx - MAX_RADIUS)
        x_max = min(x_dim, cx + MAX_RADIUS + 1)
        y_min = max(0, cy - MAX_RADIUS)
        y_max = min(y_dim, cy + MAX_RADIUS + 1)

        score = 0
        for y in range(y_min, y_max):
            dy2 = (y - cy) * (y - cy)
            for x in range(x_min, x_max):
                dx2 = (x - cx) * (x - cx)
                if dx2 + dy2 <= radius_sq:
                    if pattern_mask[cz, y, x]:
                        score += 1

        all_scores.append({
            'x': cx,
            'y': cy,
            'z': cz,
            'score': score
        })

    if len(all_scores) > 0:
        # Sort by true pattern count (descending)
        all_scores.sort(key=lambda s: s['score'], reverse=True)
        top_positions = all_scores[:top_k]
        return top_positions

    return []


# ============================================================================
# 6. Compute Accuracy (+ radius-aware IoU "Accuracy" column)
# ============================================================================

def compute_euclidean_distance(model_coords, gt_coords):
    """
    Compute Euclidean distances between model and ground truth coordinates.
    
    For each model point, find the closest GT point and compute the Euclidean distance.
    Returns statistics: mean, median, std, min, max of all distances.
    """
    # Convert to simple lists of (x, y, z)
    model_points = [(int(c['x']), int(c['y']), int(c['z'])) for c in model_coords]
    gt_points    = [(int(c['x']), int(c['y']), int(c['z'])) for c in gt_coords]

    model_count = len(model_points)
    gt_count    = len(gt_points)

    if model_count == 0 or gt_count == 0:
        return {
            'model_count': model_count,
            'gt_count': gt_count,
            'euclidean_distances': [],
            'mean_euclidean': 0.0,
            'median_euclidean': 0.0,
            'std_euclidean': 0.0,
            'min_euclidean': 0.0,
            'max_euclidean': 0.0,
        }

    euclidean_distances = []  # Store distances for each model point to closest GT

    # For each model point, find the closest GT point
    for mx, my, mz in model_points:
        min_dist = float('inf')
        
        for gx, gy, gz in gt_points:
            dx = mx - gx
            dy = my - gy
            dz = mz - gz
            dist = np.sqrt(dx*dx + dy*dy + dz*dz)
            
            if dist < min_dist:
                min_dist = dist
        
        euclidean_distances.append(min_dist)

    # Compute Euclidean distance statistics
    mean_euclidean = np.mean(euclidean_distances)
    median_euclidean = np.median(euclidean_distances)
    std_euclidean = np.std(euclidean_distances)
    min_euclidean = np.min(euclidean_distances)
    max_euclidean = np.max(euclidean_distances)

    return {
        'model_count': model_count,
        'gt_count': gt_count,
        'euclidean_distances': euclidean_distances,
        'mean_euclidean': mean_euclidean,
        'median_euclidean': median_euclidean,
        'std_euclidean': std_euclidean,
        'min_euclidean': min_euclidean,
        'max_euclidean': max_euclidean,
    }


# ============================================================================
# 7. Main Execution Loop - ALL TEXTURES AND ALL SCALES
# ============================================================================
if __name__ == "__main__":
    print("="*80)
    print("MULTI-TEXTURE EUCLIDEAN DISTANCE EXPERIMENT")
    print("="*80)
    print("\nThis will process ALL TEXTURES with ALL SCALES:")
    print(f"  Textures: {TEXTURES}")
    print(f"  Scales: {len(SCALES)} configurations")
    print("\nFor each combination:")
    print("  1. Create ground truth (synthetic)")
    print("  2. Generate subgraphs")
    print("  3. Train model")
    print("  4. Extract coordinates from model and GT")
    print("  5. Compute Euclidean distance (Mean, Median, Std)")
    print("="*80)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}")
    print(f"Total experiments: {len(TEXTURES)} textures × {len(SCALES)} scales = {len(TEXTURES) * len(SCALES)} runs\n")

    # Store all results
    all_results = []
    
    # Store results per texture for summary
    texture_results = {texture: [] for texture in TEXTURES}
    
    # Store results per scale for summary
    scale_results = {scale[0]: [] for scale in SCALES}

    experiment_count = 0
    total_experiments = len(TEXTURES) * len(SCALES)

    for texture_type in TEXTURES:
        print(f"\n{'*'*80}")
        print(f"TEXTURE: {texture_type.upper()}")
        print(f"{'*'*80}")
        
        # Create texture-specific output directory
        texture_dir = os.path.join(BASE_DIR, texture_type)
        os.makedirs(texture_dir, exist_ok=True)
        
        texture_scale_results = []

        for scale_name, x_scale, y_scale, z_scale in SCALES:
            experiment_count += 1
            print(f"\n{'#'*70}")
            print(f"[{experiment_count}/{total_experiments}] {texture_type} - {scale_name} (X={x_scale}x, Y={y_scale}x, Z={z_scale}x)")
            print(f"{'#'*70}\n")

            try:
                # 1. Create ground truth data
                data, gt_filepath = create_groundtruth(texture_type, scale_name, x_scale, y_scale, z_scale, texture_dir)

                # 2. Create subgraphs
                graphs, subgraphs_filepath = create_subgraphs(data, texture_type, scale_name, texture_dir)

                # 3. Filter graphs
                min_nodes = 50  # Reduced for smaller textures
                filtered_graphs = [g for g in graphs if g.x.shape[0] >= min_nodes]
                print(f"\n  Filtered to {len(filtered_graphs)} graphs (min {min_nodes} nodes)")

                if len(filtered_graphs) == 0:
                    print(f"  ⚠ No graphs after filtering! Skipping {texture_type} - {scale_name}...")
                    continue

                # 4. Attach GT scores to graphs for supervised training
                print(f"\n  Attaching GT scores to graphs...")
                filtered_graphs = attach_gt_scores_to_graphs(data, filtered_graphs)

                # 5. Train model
                print(f"\n  Training model for {texture_type} - {scale_name}...")
                in_channels = max([g.x.shape[1] for g in filtered_graphs])
                edge_dim = 1 if any(hasattr(g, 'edge_attr') and g.edge_attr is not None for g in filtered_graphs) else None

                model = ContrastiveGAT(
                    in_channels=in_channels,
                    hidden_channels=32,
                    projection_dim=16,
                    num_heads=4,
                    dropout=0.1,
                    edge_dim=edge_dim
                ).to(device)

                subsampled_graphs = filtered_graphs[::2] if len(filtered_graphs) > 20000 else filtered_graphs
                model, losses = train_contrastive_model(
                    model=model,
                    graphs=subsampled_graphs,
                    device=device,
                    epochs=5,  # Adjusted for faster execution
                    batch_size=32,
                    lr=0.01,
                    gradient_accumulation_steps=4
                )

                # Save model
                model_filepath = os.path.join(texture_dir, f'model_{texture_type}_{scale_name}.pt')
                model_info = {
                    'model_state_dict': model.state_dict(),
                    'in_channels': in_channels,
                    'hidden_channels': 32,
                    'projection_dim': 16,
                    'num_heads': 4,
                    'dropout': 0.1,
                    'edge_dim': edge_dim,
                    'losses': losses
                }
                torch.save(model_info, model_filepath)
                print(f"  ✓ Saved model to: {model_filepath}")

                # 6. Run GAT model to extract top positions
                print(f"\n  Running GAT model to extract top positions...")
                model_coords = find_max_interaction_positions(model, filtered_graphs, device, top_k=TOP_K)

                # Save model_coords as pickle
                model_coords_filepath = os.path.join(texture_dir, f'model_coords_{texture_type}_{scale_name}.pkl')
                with open(model_coords_filepath, 'wb') as f:
                    pickle.dump(model_coords, f)
                print(f"  ✓ Saved model coordinates to: {model_coords_filepath}")

                # Save top_positions_result as numpy array
                top_positions_array = np.array([[c['x'], c['y'], c['z']] for c in model_coords], dtype=np.int32)
                top_positions_npy_filepath = os.path.join(texture_dir, f'top_positions_{texture_type}_{scale_name}.npy')
                np.save(top_positions_npy_filepath, top_positions_array)
                print(f"  ✓ Saved top positions result (npy) to: {top_positions_npy_filepath}")

                # 7. Extract ground-truth coordinates from data
                print(f"\n  Extracting coordinates from ground truth...")
                gt_coords = extract_gt_coordinates_from_data(data, filtered_graphs, top_k=TOP_K)
                gt_coords_filepath = os.path.join(texture_dir, f'gt_coords_{texture_type}_{scale_name}.pkl')
                with open(gt_coords_filepath, 'wb') as f:
                    pickle.dump(gt_coords, f)
                print(f"  ✓ Saved GT coordinates to: {gt_coords_filepath}")

                # 8. Compute Euclidean distance between model and GT coordinates
                print(f"\n  Computing Euclidean distance between model and GT coordinates...")
                euc_result = compute_euclidean_distance(model_coords, gt_coords)
                
                result_entry = {
                    'Texture': texture_type,
                    'Scale Name': scale_name,
                    'Scale': f'{x_scale}x,{y_scale}x,{z_scale}x',
                    'GT File': os.path.basename(gt_filepath),
                    'Model Points': euc_result['model_count'],
                    'GT Points': euc_result['gt_count'],
                    'Mean Euclidean': euc_result['mean_euclidean'],
                    'Median Euclidean': euc_result['median_euclidean'],
                    'Std Euclidean': euc_result['std_euclidean'],
                    'Min Euclidean': euc_result['min_euclidean'],
                    'Max Euclidean': euc_result['max_euclidean'],
                    'Euclidean Distances': euc_result['euclidean_distances'],  # Store full list for analysis
                }
                
                all_results.append(result_entry)
                texture_results[texture_type].append(result_entry)
                scale_results[scale_name].append(result_entry)
                
                print(
                    f"  ✓ Euclidean Distance: "
                    f"Mean={euc_result['mean_euclidean']:.4f}, "
                    f"Median={euc_result['median_euclidean']:.4f}, "
                    f"Std={euc_result['std_euclidean']:.4f}, "
                    f"Min={euc_result['min_euclidean']:.4f}, "
                    f"Max={euc_result['max_euclidean']:.4f}"
                )

                # Clean up memory
                del data, graphs, filtered_graphs, model
                if device.type == 'cuda':
                    torch.cuda.empty_cache()

            except Exception as e:
                print(f"  ❌ Error processing {texture_type} - {scale_name}: {str(e)}")
                import traceback
                traceback.print_exc()
                continue

        # Save per-texture results
        if texture_results[texture_type]:
            texture_df = pd.DataFrame(texture_results[texture_type])
            texture_results_path = os.path.join(texture_dir, f'results_{texture_type}.csv')
            texture_df.to_csv(texture_results_path, index=False)
            print(f"\n  ✓ Saved {texture_type} results to: {texture_results_path}")

    # ============================================================================
    # 8. Create Final Euclidean Distance Results Tables
    # ============================================================================
    print(f"\n{'='*80}")
    print("EUCLIDEAN DISTANCE RESULTS (ALL TEXTURES × ALL SCALES)")
    print(f"{'='*80}\n")

    if all_results:
        results_df = pd.DataFrame(all_results)
        
        # Format numeric columns for display
        display_df = results_df.copy()
        for col in ['Mean Euclidean', 'Median Euclidean', 'Std Euclidean', 'Min Euclidean', 'Max Euclidean']:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}")
        
        # Remove Euclidean Distances list from display
        display_cols = [c for c in display_df.columns if c != 'Euclidean Distances']
        print(display_df[display_cols].to_string(index=False))

        # Save detailed results table (without the list column for CSV)
        save_df = results_df.drop(columns=['Euclidean Distances'])
        results_filepath = os.path.join(BASE_DIR, 'all_textures_all_scales_euclidean_results.csv')
        save_df.to_csv(results_filepath, index=False)
        print(f"\n✓ Detailed results saved to: {results_filepath}")

    # ============================================================================
    # 9. Create Euclidean Distance Summary Tables
    # ============================================================================
    print(f"\n{'='*80}")
    print("EUCLIDEAN DISTANCE SUMMARY TABLES")
    print(f"{'='*80}")

    # --- Euclidean Distance by Texture ---
    print(f"\n{'─'*60}")
    print("EUCLIDEAN DISTANCE BY TEXTURE (Mean, Median, Std)")
    print(f"{'─'*60}\n")

    texture_avg_list = []
    for texture in TEXTURES:
        if texture_results[texture]:
            tex_data = texture_results[texture]
            avg_entry = {
                'Texture': texture,
                'Num Experiments': len(tex_data),
                'Mean Euclidean': np.mean([r['Mean Euclidean'] for r in tex_data]),
                'Median Euclidean': np.mean([r['Median Euclidean'] for r in tex_data]),
                'Std Euclidean': np.mean([r['Std Euclidean'] for r in tex_data]),
                'Min Euclidean': np.min([r['Min Euclidean'] for r in tex_data]),
                'Max Euclidean': np.max([r['Max Euclidean'] for r in tex_data]),
            }
            texture_avg_list.append(avg_entry)

    if texture_avg_list:
        texture_avg_df = pd.DataFrame(texture_avg_list)
        
        # Format for display
        display_texture_avg = texture_avg_df.copy()
        for col in ['Mean Euclidean', 'Median Euclidean', 'Std Euclidean', 'Min Euclidean', 'Max Euclidean']:
            display_texture_avg[col] = display_texture_avg[col].apply(lambda x: f"{x:.4f}")
        
        print(display_texture_avg.to_string(index=False))
        
        texture_avg_path = os.path.join(BASE_DIR, 'average_by_texture.csv')
        texture_avg_df.to_csv(texture_avg_path, index=False)
        print(f"\n✓ Saved to: {texture_avg_path}")

    # --- Euclidean Distance by Scale ---
    print(f"\n{'─'*60}")
    print("EUCLIDEAN DISTANCE BY SCALE (Mean, Median, Std)")
    print(f"{'─'*60}\n")

    scale_avg_list = []
    for scale_name, x_s, y_s, z_s in SCALES:
        if scale_results[scale_name]:
            sc_data = scale_results[scale_name]
            avg_entry = {
                'Scale': scale_name,
                'Scale Config': f'{x_s}x,{y_s}y,{z_s}z',
                'Num Experiments': len(sc_data),
                'Mean Euclidean': np.mean([r['Mean Euclidean'] for r in sc_data]),
                'Median Euclidean': np.mean([r['Median Euclidean'] for r in sc_data]),
                'Std Euclidean': np.mean([r['Std Euclidean'] for r in sc_data]),
                'Min Euclidean': np.min([r['Min Euclidean'] for r in sc_data]),
                'Max Euclidean': np.max([r['Max Euclidean'] for r in sc_data]),
            }
            scale_avg_list.append(avg_entry)

    if scale_avg_list:
        scale_avg_df = pd.DataFrame(scale_avg_list)
        
        # Format for display
        display_scale_avg = scale_avg_df.copy()
        for col in ['Mean Euclidean', 'Median Euclidean', 'Std Euclidean', 'Min Euclidean', 'Max Euclidean']:
            display_scale_avg[col] = display_scale_avg[col].apply(lambda x: f"{x:.4f}")
        
        print(display_scale_avg.to_string(index=False))
        
        scale_avg_path = os.path.join(BASE_DIR, 'average_by_scale.csv')
        scale_avg_df.to_csv(scale_avg_path, index=False)
        print(f"\n✓ Saved to: {scale_avg_path}")

    # --- Overall Euclidean Distance Statistics ---
    print(f"\n{'─'*60}")
    print("OVERALL EUCLIDEAN DISTANCE (all textures × all scales)")
    print(f"{'─'*60}\n")

    if all_results:
        overall_avg = {
            'Total Experiments': len(all_results),
            'Overall Mean Euclidean': np.mean([r['Mean Euclidean'] for r in all_results]),
            'Overall Median Euclidean': np.mean([r['Median Euclidean'] for r in all_results]),
            'Overall Std Euclidean': np.mean([r['Std Euclidean'] for r in all_results]),
            'Overall Min Euclidean': np.min([r['Min Euclidean'] for r in all_results]),
            'Overall Max Euclidean': np.max([r['Max Euclidean'] for r in all_results]),
        }
        
        print(f"  Total Experiments: {overall_avg['Total Experiments']}")
        print(f"  Overall Mean Euclidean: {overall_avg['Overall Mean Euclidean']:.4f}")
        print(f"  Overall Median Euclidean: {overall_avg['Overall Median Euclidean']:.4f}")
        print(f"  Overall Std Euclidean: {overall_avg['Overall Std Euclidean']:.4f}")
        print(f"  Overall Min Euclidean: {overall_avg['Overall Min Euclidean']:.4f}")
        print(f"  Overall Max Euclidean: {overall_avg['Overall Max Euclidean']:.4f}")
        
        # Save overall average
        overall_avg_df = pd.DataFrame([overall_avg])
        overall_avg_path = os.path.join(BASE_DIR, 'overall_average.csv')
        overall_avg_df.to_csv(overall_avg_path, index=False)
        print(f"\n✓ Saved to: {overall_avg_path}")

    # --- Pivot Table: Mean Euclidean Distance (Texture × Scale) ---
    print(f"\n{'─'*60}")
    print("PIVOT TABLE: Mean Euclidean Distance (Texture × Scale)")
    print(f"{'─'*60}\n")

    if all_results:
        pivot_data = []
        for r in all_results:
            pivot_data.append({
                'Texture': r['Texture'],
                'Scale': r['Scale Name'],
                'Mean Euclidean': r['Mean Euclidean']
            })
        
        pivot_df = pd.DataFrame(pivot_data)
        pivot_table = pivot_df.pivot(index='Texture', columns='Scale', values='Mean Euclidean')
        
        # Reorder columns by scale order
        scale_order = [s[0] for s in SCALES if s[0] in pivot_table.columns]
        pivot_table = pivot_table[scale_order]
        
        # Format for display
        display_pivot = pivot_table.applymap(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")
        print(display_pivot.to_string())
        
        pivot_path = os.path.join(BASE_DIR, 'pivot_mean_euclidean_texture_scale.csv')
        pivot_table.to_csv(pivot_path)
        print(f"\n✓ Saved to: {pivot_path}")

    # ============================================================================
    # 10. EUCLIDEAN DISTANCE VISUALIZATION
    # ============================================================================
    print(f"\n{'='*80}")
    print("GENERATING EUCLIDEAN DISTANCE GRAPHS")
    print(f"{'='*80}")

    # Import matplotlib for plotting
    import matplotlib.pyplot as plt

    # Prepare data for graphs
    all_distances_by_texture = {texture: [] for texture in TEXTURES}
    all_distances_by_scale = {scale[0]: [] for scale in SCALES}
    all_euclidean_distances = []
    
    for texture in TEXTURES:
        if texture_results[texture]:
            for r in texture_results[texture]:
                all_distances_by_texture[texture].extend(r['Euclidean Distances'])
                all_euclidean_distances.extend(r['Euclidean Distances'])
    
    for scale_name, _, _, _ in SCALES:
        if scale_results[scale_name]:
            for r in scale_results[scale_name]:
                all_distances_by_scale[scale_name].extend(r['Euclidean Distances'])

    # ============================================================================
    # 11. VISUALIZATION - Euclidean Distance Graphs
    # ============================================================================
    print(f"\n{'='*80}")
    print("GENERATING EUCLIDEAN DISTANCE GRAPHS")
    print(f"{'='*80}")

    # --- Graph 1: Bar Chart - Mean Euclidean Distance by Texture ---
    if texture_avg_list:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Plot 1: Mean, Median, Std by Texture
        ax1 = axes[0, 0]
        textures = [e['Texture'] for e in texture_avg_list]
        means = [e['Mean Euclidean'] for e in texture_avg_list]
        medians = [e['Median Euclidean'] for e in texture_avg_list]
        stds = [e['Std Euclidean'] for e in texture_avg_list]
        
        x = np.arange(len(textures))
        width = 0.25
        
        ax1.bar(x - width, means, width, label='Mean', color='steelblue', alpha=0.8)
        ax1.bar(x, medians, width, label='Median', color='darkorange', alpha=0.8)
        ax1.bar(x + width, stds, width, label='Std', color='forestgreen', alpha=0.8)
        
        ax1.set_xlabel('Texture', fontsize=12)
        ax1.set_ylabel('Euclidean Distance', fontsize=12)
        ax1.set_title('Euclidean Distance Statistics by Texture', fontsize=14, fontweight='bold')
        ax1.set_xticks(x)
        ax1.set_xticklabels(textures, rotation=45, ha='right')
        ax1.legend()
        ax1.grid(axis='y', alpha=0.3)
        
        # Plot 2: Box Plot by Texture
        ax2 = axes[0, 1]
        texture_data = [all_distances_by_texture[t] for t in TEXTURES if len(all_distances_by_texture[t]) > 0]
        texture_labels = [t for t in TEXTURES if len(all_distances_by_texture[t]) > 0]
        
        if texture_data:
            bp = ax2.boxplot(texture_data, labels=texture_labels, patch_artist=True)
            colors = plt.cm.Set3(np.linspace(0, 1, len(texture_data)))
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)
            ax2.set_xlabel('Texture', fontsize=12)
            ax2.set_ylabel('Euclidean Distance', fontsize=12)
            ax2.set_title('Euclidean Distance Distribution by Texture', fontsize=14, fontweight='bold')
            ax2.tick_params(axis='x', rotation=45)
            ax2.grid(axis='y', alpha=0.3)
        
        # Plot 3: Mean Euclidean Distance by Scale
        ax3 = axes[1, 0]
        if scale_avg_list:
            scales = [e['Scale'] for e in scale_avg_list]
            scale_means = [e['Mean Euclidean'] for e in scale_avg_list]
            scale_stds = [e['Std Euclidean'] for e in scale_avg_list]
            
            x_scales = np.arange(len(scales))
            ax3.bar(x_scales, scale_means, yerr=scale_stds, capsize=3, color='teal', alpha=0.7, ecolor='darkred')
            ax3.set_xlabel('Scale', fontsize=12)
            ax3.set_ylabel('Mean Euclidean Distance', fontsize=12)
            ax3.set_title('Mean Euclidean Distance by Scale (with Std)', fontsize=14, fontweight='bold')
            ax3.set_xticks(x_scales)
            ax3.set_xticklabels(scales, rotation=45, ha='right')
            ax3.grid(axis='y', alpha=0.3)
        
        # Plot 4: Histogram of All Euclidean Distances
        ax4 = axes[1, 1]
        if len(all_euclidean_distances) > 0:
            ax4.hist(all_euclidean_distances, bins=30, color='purple', alpha=0.7, edgecolor='black')
            ax4.axvline(np.mean(all_euclidean_distances), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(all_euclidean_distances):.2f}')
            ax4.axvline(np.median(all_euclidean_distances), color='orange', linestyle='--', linewidth=2, label=f'Median: {np.median(all_euclidean_distances):.2f}')
            ax4.set_xlabel('Euclidean Distance', fontsize=12)
            ax4.set_ylabel('Frequency', fontsize=12)
            ax4.set_title('Distribution of All Euclidean Distances', fontsize=14, fontweight='bold')
            ax4.legend()
            ax4.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        
        # Save figure
        fig_path = os.path.join(BASE_DIR, 'euclidean_distance_analysis.png')
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f"\n✓ Saved figure to: {fig_path}")
        plt.show()

    # --- Graph 2: Heatmap of Mean Euclidean Distance (Texture × Scale) ---
    if all_results and len(pivot_table) > 0:
        # Create Heatmap using the pivot_table already created above
        fig, ax = plt.subplots(figsize=(12, 6))
        im = ax.imshow(pivot_table.values, cmap='YlOrRd', aspect='auto')
        
        ax.set_xticks(np.arange(len(pivot_table.columns)))
        ax.set_yticks(np.arange(len(pivot_table.index)))
        ax.set_xticklabels(pivot_table.columns, rotation=45, ha='right')
        ax.set_yticklabels(pivot_table.index)
        
        # Add colorbar
        cbar = ax.figure.colorbar(im, ax=ax)
        cbar.ax.set_ylabel('Mean Euclidean Distance', rotation=-90, va="bottom")
        
        # Add text annotations
        for i in range(len(pivot_table.index)):
            for j in range(len(pivot_table.columns)):
                val = pivot_table.iloc[i, j]
                if pd.notna(val):
                    text = ax.text(j, i, f'{val:.2f}', ha='center', va='center', color='black', fontsize=9)
        
        ax.set_title('Mean Euclidean Distance Heatmap (Texture × Scale)', fontsize=14, fontweight='bold')
        ax.set_xlabel('Scale')
        ax.set_ylabel('Texture')
        
        plt.tight_layout()
        
        heatmap_path = os.path.join(BASE_DIR, 'euclidean_heatmap.png')
        plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
        print(f"\n✓ Saved heatmap to: {heatmap_path}")
        plt.show()

    print(f"\n{'='*80}")
    print("MULTI-TEXTURE EUCLIDEAN DISTANCE EXPERIMENT COMPLETE!")
    print(f"{'='*80}")
    print(f"\nAll results saved to: {BASE_DIR}")
    print(f"\nGenerated files:")
    print(f"  - all_textures_all_scales_euclidean_results.csv (detailed results)")
    print(f"  - average_by_texture.csv (Euclidean distance statistics)")
    print(f"  - average_by_scale.csv (Euclidean distance statistics)")
    print(f"  - overall_average.csv (Overall Euclidean statistics)")
    print(f"  - pivot_mean_euclidean_texture_scale.csv (Pivot table)")
    print(f"  - euclidean_distance_analysis.png (Visualization)")
    print(f"  - euclidean_heatmap.png (Heatmap)")
    print(f"  - Per-texture folders with individual results and models")
    print(f"{'='*80}\n")
